# Reproduce the paper in Google Colab

Use a **T4 GPU** if available. Upload the repository ZIP when prompted, or clone it into `/content/waste-language-vision-reproduction`. Datasets remain in your Google Drive. Run all cells in order.

In [ ]:
from pathlib import Path
from google.colab import files
import zipfile
REPO = Path("/content/waste-language-vision-reproduction")
if not REPO.exists():
    print("Upload waste-language-vision-reproduction.zip")
    uploaded = files.upload()
    archive = next((x for x in uploaded if x.endswith(".zip")), None)
    assert archive is not None, "Please upload the GitHub-ready ZIP"
    with zipfile.ZipFile(archive) as z: z.extractall("/content")
assert REPO.is_dir(), REPO
print("Repository:", REPO)
if (REPO/'evidence.zip').is_file() and not (REPO/'measurements/COMPRESSED_FILES.json').exists():
    with zipfile.ZipFile(REPO/'evidence.zip') as z:
        z.extractall(REPO)


In [ ]:
%pip -q install -e /content/waste-language-vision-reproduction


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## Dataset paths
Set these to the extracted category-folder roots in your Drive. If your files are still ZIPs, extract them to `/content` first.

In [ ]:
from pathlib import Path
REALWASTE = Path("/content/drive/MyDrive/RealWaste")  # CHANGE ME
TRASHNET = Path("/content/drive/MyDrive/dataset-resized")  # CHANGE ME
assert REALWASTE.is_dir(), REALWASTE
assert TRASHNET.is_dir(), TRASHNET


## Zero-shot: CLIP, SigLIP, OpenCLIP
This runs all nine predefined prompt configurations on both datasets and checks the paper split manifests.

In [ ]:
import subprocess, sys
REPO = "/content/waste-language-vision-reproduction"
for dataset, root in [("realwaste", REALWASTE), ("trashnet", TRASHNET)]:
    for backbone in ["clip", "siglip", "openclip"]:
        subprocess.run([sys.executable, f"{REPO}/scripts/run_zero_shot.py", "--dataset", dataset, "--data", str(root), "--backbone", backbone, "--reference-split", f"{REPO}/reference/splits/{dataset}.csv", "--out", f"{REPO}/results/zero_shot/{dataset}/{backbone}", "--batch", "16", "--cache-dir", f"{REPO}/results/cache/{dataset}"], check=True)


## Few-shot: 30 paired support draws
This may take longer than the zero-shot experiment, but reuses cached CLIP image features within each output directory.

In [ ]:
for dataset, root in [("realwaste", REALWASTE), ("trashnet", TRASHNET)]:
    subprocess.run([sys.executable, f"{REPO}/scripts/run_few_shot.py", "--dataset", dataset, "--data", str(root), "--reference-split", f"{REPO}/reference/splits/{dataset}.csv", "--out", f"{REPO}/results/few_shot/{dataset}", "--shots", "1", "2", "4", "8", "16", "--repeats", "30", "--batch", "16", "--cache-dir", f"{REPO}/results/cache/{dataset}"], check=True)


## Original linguistic controls and overlap sensitivity
These reproduce the earlier five-condition and three-condition experiments. The candidate-exclusion analysis is **not** proof of duplicate-free datasets.


In [ ]:
for dataset, root in [("realwaste", REALWASTE), ("trashnet", TRASHNET)]:
    for protocol, nboot in [("experiment8", "2000"), ("linguistic", "5000")]:
        subprocess.run([sys.executable, f"{REPO}/scripts/run_controls.py", "--dataset", dataset, "--data", str(root), "--protocol", protocol, "--reference-split", f"{REPO}/reference/splits/{dataset}.csv", "--out", f"{REPO}/results/controls/{dataset}/{protocol}", "--bootstrap", nboot, "--cache-dir", f"{REPO}/results/cache/{dataset}"], check=True)
    subprocess.run([sys.executable, f"{REPO}/scripts/sensitivity_analysis.py", "--dataset", dataset, "--predictions", f"{REPO}/results/controls/{dataset}/linguistic/predictions.csv", "--flags", f"{REPO}/reference/audit/flagged_pairs.csv", "--out", f"{REPO}/paper_outputs/sensitivity_{dataset}.csv"], check=True)


## Generate paper-ready tables, figures and compare against archived results

In [ ]:
subprocess.run([sys.executable, f"{REPO}/scripts/make_paper_tables.py", "--results", f"{REPO}/results", "--out", f"{REPO}/paper_outputs"], check=True)
subprocess.run([sys.executable, f"{REPO}/scripts/verify_reference.py", "--results", f"{REPO}/results", "--reference", f"{REPO}/reference"], check=True)


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive("/content/paper_reproduction_outputs", "zip", f"{REPO}/paper_outputs")
files.download(archive)
